<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_0906_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 텍스트 분류 실습 - 20 뉴스그룹 분류

사이킷런 내부 예제 데이터(20 뉴스그룹 데이터 세트)를 이용한 텍스트 분류

* 텍스트 분류: 특정 문서의 분류를 학습 데이터를 통해 학습하여 모델을 생성, 생성된 학습 모델을 이용해 다른 모델의 분류 예측

- fetch_20newsgroups() API를 이용한 데이터 제공

- 텍스트 정규화 -> 피처 벡터화 -> 머신러닝 알고리즘(학습/예측/평가)

- 피처 벡터화를 위한 파라미터, GridSearchCV 기반의 하이퍼 파라미터 튜닝, 사이킷런의 Pipeline 객체 -> 피처 벡터화 파라미터 튜닝, GridSearchCV 기반의 하이퍼 파라미터 튜닝







# 텍스트 정규화

- fetch_20newsgroups(): Bunch 객체(파이썬 딕셔너리와 유서) 반환

In [77]:
from sklearn.datasets import fetch_20newsgroups

news_data=fetch_20newsgroups(subset='all', random_state=156)

In [78]:
print(news_data.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


- 'filenames': fetch_20newsgroups() API가 인터넷에서 내려받아 로컬 컴퓨터에 저장하는 디렉터리, 파일명



In [79]:
# Target 클래스의 구성 확인

import pandas as pd

print('target 클래스의 값과 분포도 \n', pd.Series(news_data.target).value_counts().sort_index())
print('target 클래스의 이름들 \n', news_data.target_names)

target 클래스의 값과 분포도 
 0     799
1     973
2     985
3     982
4     963
5     988
6     975
7     990
8     996
9     994
10    999
11    991
12    984
13    990
14    987
15    997
16    910
17    940
18    775
19    628
Name: count, dtype: int64
target 클래스의 이름들 
 ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


- Target 클래스 값 0~19 (20개) 구성


In [80]:
# 개별 데이터의 텍스트 구성 확인

print(news_data.data[0])

From: egreen@east.sun.com (Ed Green - Pixel Cruncher)
Subject: Re: Observation re: helmets
Organization: Sun Microsystems, RTP, NC
Lines: 21
Distribution: world
Reply-To: egreen@east.sun.com
NNTP-Posting-Host: laser.east.sun.com

In article 211353@mavenry.altcit.eskimo.com, maven@mavenry.altcit.eskimo.com (Norman Hamer) writes:
> 
> The question for the day is re: passenger helmets, if you don't know for 
>certain who's gonna ride with you (like say you meet them at a .... church 
>meeting, yeah, that's the ticket)... What are some guidelines? Should I just 
>pick up another shoei in my size to have a backup helmet (XL), or should I 
>maybe get an inexpensive one of a smaller size to accomodate my likely 
>passenger? 

If your primary concern is protecting the passenger in the event of a
crash, have him or her fitted for a helmet that is their size.  If your
primary concern is complying with stupid helmet laws, carry a real big
spare (you can put a big or small head in a big helmet, bu

- 텍스트 데이터: 뉴스그룹 기사 내용, 뉴스그룹 제목, 작성자, 소속, 이메일 등

- 내용 제외 타 정보 제거

(헤더, 푸터 정보들이 분류의 Target 클래스 값과 유사한 데이터 / 피처들을 포함할 시, 높은 예측 성능)

- remove 파라미터: 뉴스그룹 기사의 헤더, 푸터 등 제거

- subset 파라미터: 학습 데이터 세트, 테스트 데이터 세트 분리




In [81]:
# 순수한 텍스트만으로 구성된 기사 내용이 어떤 뉴스그룹에 속하는지 분류
from sklearn.datasets import fetch_20newsgroups

# subset='train'으로 학습용 데이터만 추출, remove=('headers', 'footers', 'quotes')로 내용만 추출
train_news=fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), random_state=156)
X_train=train_news.data
y_train=train_news.target

# subset='test'으로 테스트 데이터만 추출, remove=('headers', 'footers', 'quotes')로 내용만 추출
test_news=fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), random_state=156)
X_test=test_news.data
y_test=test_news.target
print('학습 데이터 크기 {0}, 테스트 데이터 크기 {1}'.format(len(train_news.data),
                                             len(test_news.data)))


학습 데이터 크기 11314, 테스트 데이터 크기 7532


# 피처 벡터화 변환과 머신러닝 모델 학습/예측/평가

학습 데이터: 11314개의 뉴스그룹 문서 리스트 형태

테스트 데이터: 7532개의 문서 리스트 형태

- CountVectorizer: 학습 데이터의 텍스트 피처 벡터화

(테스트 데이터에서 CountVectorizer 적용 시,

학습 데이터를 이용해 fit()이 수행된 CountVectorizer 객체를 이용해 테스트 데이터 반환(transform)

-> 학습 시 설정된 CountVectorizer의 피처 개수 = 테스트 데이터를 CountVectorizer로 변환할 피처 개수)

- cnt_vect.transform(): 테스트 데이터의 피처 벡터화

(fit_transform()을 테스트 데이터 세트에 적용 시,

테스트 데이터 기반으로 다시 CountVectorizer가 fit() 수행, transform() 수행

-> 학습 시 사용된 피처 개수, 예측 시 사용할 피처 개수 달라짐)



In [82]:
from sklearn.feature_extraction.text import CountVectorizer

# Count Vectorization으로 피처 벡터화 변환 수행.
cnt_vect=CountVectorizer()
cnt_vect.fit(X_train)
X_train_cnt_vect=cnt_vect.transform(X_train)

# 학습 데이터로 fit()된 CountVectorizer를 이용해 테스트 데이터를 피처 벡터화 변환 수행.
X_test_cnt_vect=cnt_vect.transform(X_test)

print('학습 데이터 텍스트의 CountVectorizer Shape:', X_train_cnt_vect.shape)

학습 데이터 텍스트의 CountVectorizer Shape: (11314, 101631)


- 학습 데이터를 CountVectorizer로 피처 추출 결과

: 11314개의 문서에서 피처(단어)가 101631개 생성



In [83]:
# 피처 벡터화된 데이터에 로지스틱 회귀 적용 -> 뉴스그룹에 대한 분류 예측
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# LogisticRegression을 이용하여 학습/예측/평가 수행.
lr_clf=LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_cnt_vect, y_train)
pred=lr_clf.predict(X_test_cnt_vect)
print('CountVectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test,pred)))

CountVectorized Logistic Regression의 예측 정확도는 0.617


- Count 기반 피처 벡터화가 적용된 데이터 세트의 로지스틱 회귀의 예측 정확도

= 0.616



In [84]:
# Count 기반 TF-IDF 기반 벡터화 변경 -> 예측 모델 수행
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 벡터화를 적용해 학습 데이터 세트와 테스트 데이터 세트 변환.
tfidf_vect=TfidfVectorizer()
tfidf_vect.fit(X_train)
X_train_tfidf_vect=tfidf_vect.transform(X_train)
X_test_tfidf_vect=tfidf_vect.transform(X_test)

# LogisticRegression을 이용해 학습/예측/평가 수행.
lr_clf=LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)
pred=lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))


TF-IDF Logistic Regression의 예측 정확도는 0.678


- TF-IDF가 단순 카운트 기반보다 훨씬 높은 예측 정확도 제공

(문서 내 텍스트가 많고 많은 문서를 가지는 텍스트 분석에서,

카운트 벡터화보다 TF-IDF 벡터화가 더 좋은 예측 결과 도출)





텍스트 분석에서의 머신러닝 모델의 성능 향상 방법

1. 최적의 ML 알고리즘 선택

2. 최상의 피처 전처리 수행

- 텍스트 정규화, Count/TF-IDF 기반 피처 벡터화 효과적 적용 방법이 머신러닝 성능에 큰 영향

-> 다양한 파라미터 적용

(1) TfidfVectorizer 클래스의 스톱 워드 변경

(기존 'None' -> 'english')

(2) ngram_range 변경

(기존 (1,1) -> (1,2))

(3) max_df 변경

(max_df=300으로 변경)





In [85]:
# stop words 필터링을 추가하고 ngram을 기본 (1,1)에서 (1,2)로 변경해 피처 벡터화 적용.
tfidf_vect=TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_df=300)
tfidf_vect.fit(X_train)
X_train_tfidf_vect=tfidf_vect.transform(X_train)
X_test_tfidf_vect=tfidf_vect.transform(X_test)

lr_clf=LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)
pred=lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test,pred)))

TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.690


In [86]:
# GridSearchCV를 이용한 로지스틱 회귀의 하이퍼 파라미터 최적화 수행
# 로지스틱 회귀의 C 파라미터만 변경->최적의 C값 찾기-> 최적의 C값으로 학습된 모델에서 테스트 데이터로 예측, 성능 평가

from sklearn.model_selection import GridSearchCV

# 최적 C 값 도출 튜닝 수행. CV는 3 폴드 세트로 설정.
params={ 'C':[0.01, 0.1, 1, 5, 10]}
grid_cv_lr=GridSearchCV(lr_clf, param_grid=params, cv=3, scoring='accuracy', verbose=1)
grid_cv_lr.fit(X_train_tfidf_vect, y_train)
print('Logistic Regression best C parameter :', grid_cv_lr.best_params_)

# 최적 C값으로 학습된 grid_cv로 예측 및 정확도 평가.
pred=grid_cv_lr.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

#


Fitting 3 folds for each of 5 candidates, totalling 15 fits
Logistic Regression best C parameter : {'C': 10}
TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.704


- 로지스틱 회귀의 C가 10일 때, GridSearchCV의 교차 검증 테스트 세트에서 가장 좋은 예측 성능

- 테스트 데이터 세트에 적용했을 때, 0.704 (약간 향상된 성능 수치)

# 사이킷런 파이프라인(Pipeline) 사용 및 GridSearchCV와의 결합

사이킷런의 Pipeline 클래스 이용 -> 피처 벡터화 + ML 알고리즘 학습/예측 코드 작성

*pipeline 이용 시,

- 데이터의 전처리, 머신러닝 학습 과정을 통일된 API 기준에서 처리 -> 더 직관적인 ML 모델 코드 생성

- 대용량 데이터의 피처 벡터화 결과를 별도 데이터로 저장 X,

스트림 기반에서 머신러닝 알고리즘의 데이터로 입력 -> 수행 시간 절약

- 사이킷런 파이프라인: 텍스트 기반의 피처 벡터화 + 데이터 전처리 작업 + Estimator 결합

(스케일링 or 벡터 정규화 or PCA + Estimator 결합)











In [87]:
import os
from sklearn.pipeline import Pipeline
# 위의 텍스트 분류 예제 코드를 Pipeline을 이용하여 작성
# Pipeline 객체
pipeline=Pipeline([('tfidf_vect', TfidfVectorizer(stop_words='english')),
                   ('lr_clf', LogisticRegression(random_state=156))])

- Pipeline 방식을 적용한 머신러닝 코드

1. TfidfVectorizer 객체 -> 객체 변수명: tfidf_vect + LogisticRegression 객체 -> 객체 변수명: lr_clf

= Pipeline 객체 pipeline 생성

(두 개의 객체를 파이프라인으로 연결)

2. 기존 TfidfVectorizer의 학습 데이터, 테스트 데이터에 대한 fit(), transform() 수행 -> 피처 벡터화 + LogisticRegressor의 fit(), predict() 수행 -> 머신러닝 학습, 예측

= Pipeline의 fit(), predict()

(통일 수행)



In [88]:
from sklearn.pipeline import Pipeline

# TfidfVectorizer 객체를 tfidf_vect로, LogisticRegression 객체를 lr_clf로 생성하는 Pipeline 생성
pipeline=Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_df=300)),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])

# 별도의 TfidfVectorizer 객체의 fit(), transform()과 LogisticRegression의 fit(), predict()가 필요 없음.
# pipeljne의 fit()과 predict()만으로 한꺼번에 피처 벡터화와 ML 학습/예측이 가능.
pipeline.fit(X_train, y_train)
pred=pipeline.predict(X_test)
print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

Pipeline을 통한 Logistic Regression의 예측 정확도는 0.704


- 사이킷런의 GriesSearchCV 클래스의 생성 파라미터로 Pipeline 입력

-> Pipeline 기반 하이퍼 파라미터 튜닝을 GridSearchCV 방식으로 진행

(GridSearchCV 를 이용하여 피처 벡터화를 위한 파라미터, ML 알고리즘의 하이퍼 파라미터 동시에 최적화)





예제) GriedSearchCV에 Pipeline 입력

-> TfidfVectorizer의 파라미터, Logistic Regression의 하이퍼 파라미터 최적화

- GridSearchCV에 Pipeline 입력 시,

param_grid의 입력값 설정

-> 딕셔너리 형태의 Key, Value 값

-> Value를 리스트 형태로 입력

-> Key 값이 하이퍼 파라미명+객체 변수명 제공

( ex - 'tfidf_vect__ngram_range')

-> Pipeline + GridSearchCV 적용 시,

모든 파라미터 최적화 시, 과다의 경우의 수 (시간 소모 high)





In [33]:
# Pipeline + GridSearchCV 기반 하이퍼 파라미터 튜닝 적용
# 27개의 파라미터 경우의 수 X 3개의 CV = 81번의 학습/검증 (약 24분)

from sklearn.pipeline import Pipeline

pipeline=Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english')),
    ('lr_clf', LogisticRegression())
])

# Pipeline에 기술된 각각의 객체 변수에 언더바(_) 2개를 연달아 붙여 GridSearchCV에 사용될 파라미터/하이퍼 파라미터 이름과 값을 설정.
params={'tfidf_vect__ngram_range': [(1, 1), (1, 2), (1, 3)],
        'tfidf_vect__max_df': [100, 200, 700],
        'lr_clf__C': [1, 5, 10]
}

# GridSearCV의 생성자에 Estimator가 아닌 Pipeline 객체 입력
grid_cv_pipe=GridSearchCV(pipeline, param_grid=params, cv=3, scoring='accuracy', verbose=1)
grid_cv_pipe.fit(X_train, y_train)
print(grid_cv_pipe.best_params_, grid_cv_pipe.best_score_)

pred=grid_cv_pipe.predict(X_test)
print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))



Fitting 3 folds for each of 27 candidates, totalling 81 fits


KeyboardInterrupt: 

결과:

- max_df 파라미터가 700, ngram_range 파라미터가 (1,2), C 하이퍼 파라미터 10 적용 시, 가장 좋은 검증 세트 성능 수치 도출

(이때, 정확도 약 0.702 (큰 개선 X))

시사점:

희소 행렬 기반 텍스트 분류에 자주 사용되는 머신러닝 알고리즘

(서포트 벡터머신(Support Vector Machine), 나이브 베이즈(Naive Bayes) 알고리즘)

을 이용한 모델 사용





# 5. 감성 분석

# 감성 분석 소개

감성 분석(Sentiment Analysis): 문서의 주관적인 감성/의견/감정/기분 등 파악하기 위한 방법

- 문서 내 텍스트가 나타내는 주관적인 단어, 문맥 기반 감성(Sentiment)수치 계산 방법 이용

- 긍정 감성 지수 + 부정 감성 지수

-> 긍정 감성 / 부정 감성 결정

1. 지도 학습

: 학습 데이터, 타깃 레이블 값 기반 감성 분석 학습

-> 다른 데이터의 감성 분석 예측

2. 비지도 학습

: Lexicon (감성 분석을 위한 용어, 문맥에 대한 다양한 정보 가지고 있음.)을 이용한 문서의 긍정적, 부정적 감성 여부 판단







# 지도학습 기반 감성 분석 실습 - IMDB 영화평

영화평의 텍스트 분석 -> 감성 분석 결과의 긍정 / 부정 예측 모델 생성



In [36]:
import pandas as pd

review_df=pd.read_csv('./labeledTrainData.tsv', header=0, sep="\t", quoting=3)
review_df.head(3)

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."


In [37]:
print(review_df['review'][0])

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actual feature film bit when it finally sta

- <br / > 태그

DataFrame/Series 객체에서 str 적용 -> <br / > 태그 공백으로 변경

- 숫자/특수문자

(정규 표현식([^a-zA-Z] 이용)

re.sub("[^a-zA-Z]"," ", x) 적용 -> 영어 대/소문자가 아닌 모든 문자 공란으로 변경







In [38]:
# 판다스 DataFrame에 Lambda 식을 이용한 re.sub() 적용
import re

# <br> gtml 태그는 replace 함수로 공백으로 변환
review_df['review']=review_df['review'].str.replace('<br />', ' ')

# 파이썬의 정규 표현식 모듈인 re를 이용해 영어 문자열이 아닌 문자는 모두 공백으로 변환
review_df['review']=review_df['review'].apply(lambda x: re.sub("[^a-zA-Z]", " ",x) )


In [41]:
# 결정 값 클래스인 sentiment 칼럼 별도 추출하여 결정 값 데이터 세트 생성
# 원본 데이터 세트에서 id, sentiment 칼럼 삭제하여 피처 데이터 세트 생성
# train_test_split()을 이용한 학습용, 테스트용 데이터 세트로 분리
from sklearn.model_selection import train_test_split

class_df=review_df['sentiment']
feature_df=review_df.drop(['id','sentiment'], axis=1, inplace=False)
X_train, X_test, y_train, y_test=train_test_split(feature_df, class_df, test_size=0.3, random_state=156)
X_train.shape, X_test.shape

((17500, 1), (7500, 1))

- 학습용 데이터는 17500개의 리뷰, 테스트용 데이터는 7500개의 리뷰로 구성



감상평(Review) 텍스트 피처 벡터화 -> ML 분류 알고리즘 적용 -> 예측 성능 측정

(Pipeline 객체 이용하여 한 번에 수행)

1. Count 벡터화 적용

2. TF-IDF 벡터화 적용

Classifier는 LogisticRegression 이용

이진 분류 (예측 성능 평가) -> 테스트 데이터 세트의 정확도, ROC-AUC 측정







In [42]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 스톱 워드는 English, ngram은 (1, 2)로 설정해 CoutVectorization 수행.
# LogisticRegression의 C는 10으로 설정.
pipeline=Pipeline([
    ('cnt_vect', CountVectorizer(stop_words='english', ngram_range=(1, 2) )),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))])

# Pipeline 객체를 이용해 fit(), predict()로 학습/예측 수행. predict_proba()는 roc_auc 때문에 수행.
pipeline.fit(X_train['review'], y_train)
pred=pipeline.predict(X_test['review'])
pred_probs=pipeline.predict_proba(X_test['review'])[:, 1]

print('예측 정확도는 {0:.4f}, ROC-AUC는 {1:.4f}'.format(accuracy_score(y_test, pred),
                                                 roc_auc_score(y_test, pred_probs)))



예측 정확도는 0.8861, ROC-AUC는 0.9503


In [44]:
# TF-IDF 벡터화를 적용한 예측 성능 측정
# 위의 예체 코드에서, Popeline에서 CountVectorizer를 TfidVectorizer로 변경

# 스톱 워드는 english, filtering, ngram은 (1, 2)로 설정해 TF-IDF 벡터화 수행.
# LogisticRegression의 C는 10으로 설정.
pipeline=Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=(1, 2) )),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))])

pipeline.fit(X_train['review'], y_train)
pred=pipeline.predict(X_test['review'])
pred_probs=pipeline.predict_proba(X_test['review'])[:1]



- TF-IDF 기반 피처 벡터화의 예측 성능이 조금 더 좋음.



# 비지도학습 기반 감성 분석 소개

비지도 감성 분석 (Lexicon 기반)

<-> 지도 감성 분석: 데이터 세트가 레이블 값을 가짐.

(실제로는 이러한 결정된 레이블 값을 갖지 X. -> Lexicon의 유용한 사용)

Lexicon: 감성만을 분석하기 위해 지원하는 감성 어휘 사전

단어 위치, 주변 단어, 문맥, POS(Part of Speech) 등 -> 감성 지수 결정

1. NLTK에서 제공하는 WordNet 모듈

- 시맨틱(문맥상 의미) 분석을 제공하는 어휘 사전

- 시맨틱을 프로그램적으로 인터페이스할 수 있는 다양한 방법 제공

- 각각의 품사로 구성된 개별 단어를 Synset이라는 개념을 이용해 표현

- 예측 성능이 좋지 않음.

- SentiWordNet, VADER, Pattern






# SentiWordNet을 이용한 감성 분석

NLTK 셋업 -> WordNet 서브패키지, 데이터 세트 내려받기 -> WordNet 모듈 임포트 -> 'present' 단어에 대한 Synset 추출 (synsets())

synsets(): 여러 개의 Synset 객체를 가지는 리스트 반환

ex) Synset('present.n.01' - present(의미), n(명사 품사), 01(present 의미 구분 인덱스))



In [45]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [46]:
from nltk.corpus import wordnet as wn

term='present'

# 'present'라는 단어로 wordnet의 synsets 생성.
synsets=wn.synsets(term)
print('synsets() 반환 type :', type(synsets))
print('synsets() 반환 값 개수:', len(synsets))
print('synsets() 반환 값 :', synsets)

synsets() 반환 type : <class 'list'>
synsets() 반환 값 개수: 18
synsets() 반환 값 : [Synset('present.n.01'), Synset('present.n.02'), Synset('present.n.03'), Synset('show.v.01'), Synset('present.v.02'), Synset('stage.v.01'), Synset('present.v.04'), Synset('present.v.05'), Synset('award.v.01'), Synset('give.v.08'), Synset('deliver.v.01'), Synset('introduce.v.01'), Synset('portray.v.04'), Synset('confront.v.03'), Synset('present.v.12'), Synset('salute.v.06'), Synset('present.a.01'), Synset('present.a.02')]


In [48]:
# synset 객체가 가지는 속성들
# Synset은 POS, 정의, 부명제 등으로 시맨틱적인 요소 표현
for synset in synsets:
  print('##### Synset name : ', synset.name(), '#####')
  print('POS :', synset.lexname())
  print('Definition:', synset.definition())
  print('Lemmas:', synset.lemma_names())

##### Synset name :  present.n.01 #####
POS : noun.time
Definition: the period of time that is happening now; any continuous stretch of time including the moment of speech
Lemmas: ['present', 'nowadays']
##### Synset name :  present.n.02 #####
POS : noun.possession
Definition: something presented as a gift
Lemmas: ['present']
##### Synset name :  present.n.03 #####
POS : noun.communication
Definition: a verb tense that expresses actions or states at the time of speaking
Lemmas: ['present', 'present_tense']
##### Synset name :  show.v.01 #####
POS : verb.perception
Definition: give an exhibition of to an interested audience
Lemmas: ['show', 'demo', 'exhibit', 'present', 'demonstrate']
##### Synset name :  present.v.02 #####
POS : verb.communication
Definition: bring forward and present to the mind
Lemmas: ['present', 'represent', 'lay_out']
##### Synset name :  stage.v.01 #####
POS : verb.creation
Definition: perform (a play), especially on a stage
Lemmas: ['stage', 'present', 'represen

- synset을 통해 하나의 단어가 가질 수 있는 여러 가지 시맨틱 정보를 개별 클래스로 나타냄.

WordNet으로 어떤 어휘, 다른 어휘 간의 관계를 유사도로 나타내기

path_similarity()

In [50]:
# 'tree', 'lion', 'tiger', 'cat', 'dog' 단어들의 상호 유사도
tree=wn.synset('tree.n.01')
lion=wn.synset('lion.n.01')
tiger=wn.synset('tiger.n.02')
cat=wn.synset('cat.n.01')
dog=wn.synset('dog.n.01')

entities=[tree, lion, tiger, cat, dog]
similarities=[]
entity_names=[entity.name().split(',')[0] for entity in entities]

# 단어별 synset을 반복하면서 다른 단어의 synset과 유사도를 측정합니다.
for entity in entities:
  similarity=[round(entity.path_similarity(compared_entity),2)
  for compared_entity in entities]
  similarities.append(similarity)

# 개별 단어별 synset과 다른 단어의 synset과의 유사도를 DataFrame 형태로 저장합니다.
similarity_df=pd.DataFrame(similarities, columns=entity_names, index=entity_names)
similarity_df




,tree.n.01,lion.n.01,tiger.n.02,cat.n.01,dog.n.01
tree.n.01,1.00,0.07,0.07,0.08,0.12
lion.n.01,0.07,1.00,0.33,0.25,0.17
tiger.n.02,0.07,0.33,1.00,0.25,0.17
cat.n.01,0.08,0.25,0.25,1.00,0.20
dog.n.01,0.12,0.17,0.17,0.20,1.00


- SentiWordNet은 Wordnet의 Synset과 유사한 Senti_Synset 클래스

- senti_synsets()는 Senti_Synset 클래스를 리스트 형태로 반환



In [51]:
import nltk
from nltk.corpus import sentiwordnet as swn

senti_synsets=list(swn.senti_synsets('slow'))
print('senti_synsets() 반환 type :', type(senti_synsets))
print('senti_synsets() 반환 값 개수:', len(senti_synsets))
print('senti_synsets() 반환 값:', senti_synsets)

senti_synsets() 반환 type : <class 'list'>
senti_synsets() 반환 값 개수: 11
senti_synsets() 반환 값: [SentiSynset('decelerate.v.01'), SentiSynset('slow.v.02'), SentiSynset('slow.v.03'), SentiSynset('slow.a.01'), SentiSynset('slow.a.02'), SentiSynset('dense.s.04'), SentiSynset('slow.a.04'), SentiSynset('boring.s.01'), SentiSynset('dull.s.08'), SentiSynset('slowly.r.01'), SentiSynset('behind.r.03')]


- SentiSynset 객체

(1) 감성 지수: 단어의 감성을 나타냄

(긍정 감성 지수, 부정 감성 지수)

(2) 객관성 지수: 객관성을 나타냄

(전혀 감성적 X = 객관성 지수 1, 감성 지수 0)



In [54]:
# father(아버지), fabulous(아주 멋진)의 감성 지수, 객관성 지수
import nltk
from nltk.corpus import sentiwordnet as swn

father=swn.senti_synset('father.n.01')
print('father 긍정감성 지수:', father.pos_score())
print('father 부감성 지수:', father.neg_score())
print('father 객관성 지수:', father.obj_score())
print('\n')
fabulous=swn.senti_synset('fabulous.a.01')
print('fabulous 긍정감성 지수:', fabulous.pos_score())
print('fabulous 부정감성 지수:', fabulous.neg_score())

father 긍정감성 지수: 0.0
father 부감성 지수: 0.0
father 객관성 지수: 1.0


fabulous 긍정감성 지수: 0.875
fabulous 부정감성 지수: 0.125


- father은 객관적인 단어

- fabulous는 감성 단어 (긍정 감성 지수 0.875, 부정 감성 지수 0.125)

# SentiWordNet을 이용한 영화 감상평 감성 분석

IMDB 영화 감상평 감성 분석을 SentiWordNet Lexicon 기반 수행

1. 문서를 문장 단위로 분해

2. 문장을 단어 단위로 토큰화, 품사 태깅

3. 품사 태깅된 단어 기반 synset 객체, senti_synset 객체 생성

4. senti_synset 객체에서 긍정 감성/부정 감성 지수 구하기 -> 합산

(특정 임계치 값 이상 -> 긍정 감성 / X -> 부정 감성)



In [61]:
from nltk.corpus.reader.wordnet import NOUN
# 품사 태깅 수행 내부 함수 생성
from nltk.corpus import wordnet as wn

# 간단한 NTLK PennTreebank Tag를 기반으로 WordNet 기반의 품사 Tag로 변환
def penn_to_wn(tag):
  if tag.startswith('J'):
    return wn.ADJ
  elif tag.startswith('N'):
    return wn.NOUN
  elif tag.startswith('R'):
    return wn.ADV
  elif tag.startswith('V'):
    return wn.VERB

In [58]:
# 문서 -> 문장 -> 단어 토큰 -> 품사 태깅 -> SentiSynset 클래스 생성 -> Polarity Score 합산 함수 생성
# 총 감성 지수가 0 이상인 경우 긍정 감성, 그렇지 않을 경우 부정 감성
from nltk.stem import WordNetLemmatizer
from nltk.corpus import sentiwordnet as swn
from nltk import sent_tokenize, word_tokenize, pos_tag

def swn_polarity(text):
  # 감성 지수 초기화
  sentiment=0.0
  tokens_count=0

  lemmatizer=WordNetLemmatizer()
  raw_sentences=sent_tokenize(text)
  # 분해된 문장별로 단어 토큰 -> 품사 태깅 후에 SentiSynset 생성 -> 감성 지수 합산
  for raw_sentence in raw_sentences:
    # NLTK 기반의 품사 태깅 문장 추출
    tagged_sentence=pos_tag(word_tokenize(raw_sentence))
    for word, tag in tagged_sentence:

      # WordNet 기반 품사 태깅과 어근 추출
      wn_tag=penn_to_wn(tag)
      if wn_tag not in (wn.NOUN, wn.ADJ, wn.ADV):
        continue
        lemma=lammatizer.lammatize(word, pos=wn_tag)
        if not lemma:
          continue
          # 어근을 추출한 단어와 WordNet 기반 품사 태깅을 입력해 Synset 객체를 생성.
          synsets=wn.synsets(lemma, pos=wn_tag)
          if not synsets:
            continue
          # sentiwordnet의 감성 단어 분석으로 감성 synset 추출
          # 모든 단어에 대해 긍정 감성 지수는 +로 부정 감성 지수는 -로 합산해 감성 지수 계산.
          synset=synsets[0]
          swn_synset=swn.senti_synset(synset.name())
          sentiment+=(swn_synset.pos_score()-swn_synset.neg_score())
          tokens_count+=1

        if not tokens_count:
          return 0

        # 총 score가 0 이상일 경우 긍정(Positive) 1, 그렇지 않을 경우 부정(Negative) 0 반환
        if sentiment >=0:
          return 1

        return 0

In [62]:
# swn_polarity(text)함수를 IMDB 감상평의 개별 문서에 적용 (10분)
# apply_lambda 이용
# 지도학습 기반의 감성 분석에서 생성한 review_df DataFrame 그대로 이용
# review_df의 칼럼으로 'preds' 추가
# 'sentiment' 칼럼, 정확도, 정밀도, 재현율 값 모두 측정
review_df['preds']=review_df['review'].apply(lambda x : swn_polarity(x) )
y_target=review_df['sentiment'].values
preds=review_df['preds'].values

In [ ]:
# SentiWordNet의 감성 분석 예측 성능
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score
from sklearn.metrics import recall_score, f1_score, roc_auc_score
import numpy as np

print(confusion_matrix(y_target, preds))
print("정확도:", np.round(accuracy_score(y_target, preds), 4))
print("정밀도:", np.round(precision_score(y_target, preds), 4))
print("재현율:", np.round(recall_score(y_target, preds), 4))

결과

- 정확도 약 66.13%, 재현율 약 70.91%

# VADER를 이용한 감성 분석

VADER lexicon

VADER: 소셜 미디어의 감성 분석 용도로 만들어진 룰 기반의 Lexicon

: SentimentIntensityAnalyzer 클래스

: NLTK 패키지의 서브 모듈 / 단독 패키지



In [65]:
# NLTK 서브 모듈로 SentimentIntensityAnalyzer 임포트
# IMDB 감상평의 감성 분석 수행
from nltk.sentiment.vader import SentimentIntensityAnalyzer

senti_analyzer=SentimentIntensityAnalyzer()
senti_scores=senti_analyzer.polarity_scores(review_df['review'][0])
print(senti_scores)

{'neg': 0.13, 'neu': 0.743, 'pos': 0.127, 'compound': -0.7943}


SentimentIntensityAnalyzer 객체 생성 -> 문서별 polarity_scores() 메서드 호출 -> 감성 점수 (딕셔너리 형태) 구하기 -> 감성 점수: 특정값 이상이면 긍정, X이면 부정

'neg': 부정 감성 지수

'neu': 중립 감성 지수

'pos': 긍정 감성 지수

compound: neg, neu, pos score 조합 -> -1~1 감성 지수 표현값

-> compund score 기반 부정 감성, 긍정 감성 여부 결정

(0.1 이상 -> 긍정 감성, 0.1 이하 -> 부정 감성

임계값 조절 가능)






review_df DataFrame의 apply lambda 식

-> vader_polarity() 함수 생성

(입력 파라미터: 영화 감상평 텍스트, 긍정/부정 결정 임곗값)

-> 각 문서별 감성 결과를 vader_preds라는 review_df의 새로운 칼럼으로 저장

-> VADER 예측 성능 측정

In [74]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer

def vader_polarity(review, threshold=0.1):
  analyzer=SentimentIntensityAnalyzer()
  scores=analyzer.polarity_scores(review)

  # compound 값에 기반해 threshold 입력값보다 크면 1, 그렇지 않으면 0을 반환
  agg_score=scores['compound']
  final_sentiment=1 if agg_score >= threshold else 0
  return final_sentiment

# apply_lambda 식을 이용해 레코드별로 vader_polarity()를 수행하고 결과를 'vader_preds'에 저장
review_df['vader_preds']=review_df['review'].apply(lambda x : vader_polarity(x, 0.1) )
y_target=review_df['sentiment'].values
vader_preds=review_df['vader_preds'].values

print(confusion_matrix(y_target, vader_preds))
print("정확도:", np.round(accuracy_score(y_target, vader_preds),4))
print("정밀도:", np.round(precision_score(y_target, vader_preds),4))
print("재현율:", np.round(recall_score(y_target, vader_preds),4))

[[ 6747  5753]
 [ 1858 10642]]
정확도: 0.6956
정밀도: 0.6491
재현율: 0.8514


결과

- 정확도가 SentiWorNet보다 향상

(재현율 약 85.14%로 매우 큰 향상)

한계

- 감성 사전을 이용한 감성 분석 예측 성능은 지도학습 분류 기반의 예측 성능에 비해 낮은 수준
